# Refua Peptide Design Notebook: Integrin alphaVbeta3 Cyclic Binder

> **Audience:** beginner to advanced.  
> **Goal:** design cyclic peptide candidates against an integrin target with known RGD-binding structural biology.

## Why this target is practical
- **Target:** integrin **alphaVbeta3** extracellular ligand-binding head region.
- **Complex precedent:** PDB **1L5G** includes alphaVbeta3 bound to an **RGD peptide ligand**.
- **Peptide relevance:** cyclic RGD-like peptides are a classic integrin-targeting strategy in translational oncology.

## Workflow
1. Define alphaV/beta3 target chains and approximate hotspot windows.
2. Use helper APIs to create cyclic and linear peptide design templates.
3. Fold one cyclic design and inspect binder outputs for triage.


In [ ]:
%load_ext refua_notebook
import refua_notebook

refua_notebook.activate()


In [ ]:
from refua import BinderDesigns, Complex, Protein


## Biological setup (plain language)

Integrin alphaVbeta3 recognizes RGD-like ligands. Here we design **new cyclic peptide candidates** for this ligand-binding context.

| Design input | Choice in this notebook | Why |
|---|---|---|
| Target structure context | 1L5G alphaVbeta3 + RGD peptide | Direct experimental complex precedent |
| Target chains used | Truncated alphaV and beta3 head regions | Focus on ligand-binding domains for exploratory design speed |
| Reference motif | `RGDFV` (1L5G peptide chain C) | Practical motif anchor for interpretation |
| Hotspot windows | alphaV `120..260`, beta3 `95..220` | Broad exploratory ranges around the known binding interface |


In [ ]:
INTEGRIN_AV_HEAD_1L5G = (
    "FNLDVDSPAEYSGPEGSYFGFAVDFFVPSASSRMFLLVGAPKANTTQPGIVEGGQVLKCDWSSTRRCQPIEFDATGNRDYAKDDPLEFKSHQWFGASVRSKQDKILACAPLYHWRTEMKQEREPVGTCFLQDGTKTVEYAPCRSQDIDADGQGFCQGGFSIDFTKADRVLLGGPGSFYWQGQLISDQVAEIVSKYDPNVYSIKYNNQLATRTAQAIFDDSYLGYSVAVGDFNGDGIDDFVSGVPRAARTLGMVYIYDGKNMSSLYNFTGEQMAAYFGFSVAATDINGDDYADVFIGAPLFMDRGSDGKLQEVGQVSVSLQRASGDFQTTKLNGFEVFARFGSAIAPLGDLDQDGFNDIAIAAPYGGEDKKGIVYIFNGRSTGLNAVPSQILEGQWAARSMPPSFGYSMKGATDIDKNGYPDLIVGAFGVDRAILYRARPVITVNAGLEVY"
)

INTEGRIN_B3_HEAD_1L5G = (
    "GPNICTTRGVSSCQQCLAVSPMCAWCSDEALPLGSPRCDLKENLLKDNCAPESIEFPVSEARVLEDRPLSDKGSGDSSQVTQVSPQRIALRLRPDDSKNFSIQVRQVEDYPVDIYYLMDLSYSMKDDLWSIQNLGTKLATQMRKLTSNLRIGFGAFVDKPVSPYMYISPPEALENPCYDMKTTCLPMFGYKHVLTLTDQVTRFNEEVKKQSVSRNRDAPEGGFDAIMQATVCDEKIGWRNDASHLLVFTTDAKTHIALDGRLAGIVQPNDGQCHVGSDNHYSASTTMDYPSLGLMTEKLSQKNINLIFAVTENVVNLYQNYSELIPGTTVGVLSMDSSNVLQLIVDAYGKIRSKVELEVRDLPEELSLSFNATCLNNEVIPGLKSCMGLKIGDTVSFSIEAKVRGCPQEKEKSFTIKPVG"
)

RGD_REFERENCE_1L5G = "RGDFV"
ALPHAV_BINDING_WINDOW = "120..260"
BETA3_BINDING_WINDOW = "95..220"


## Build cyclic peptide design specs with helper APIs

We use helper presets for a quick comparison:
- `BinderDesigns.disulfide_peptide(...)` to generate a cyclic-style template.
- `BinderDesigns.peptide(...)` as a linear baseline.

This notebook runs the cyclic branch first because cyclic scaffolds are common in integrin-targeted peptide programs.


In [ ]:
alpha_v = Protein(
    INTEGRIN_AV_HEAD_1L5G,
    ids="A",
    binding_types={"binding": ALPHAV_BINDING_WINDOW},
)
beta_3 = Protein(
    INTEGRIN_B3_HEAD_1L5G,
    ids="B",
    binding_types={"binding": BETA3_BINDING_WINDOW},
)

cyclic_peptide = BinderDesigns.disulfide_peptide(
    segment_lengths=(3, 5, 3),
    ids="P",
)
linear_peptide = BinderDesigns.peptide(length=9, ids="Q")

cyclic_design = Complex([alpha_v, beta_3, cyclic_peptide], name="itgavb3_cyclic_peptide_design")
linear_design = Complex([alpha_v, beta_3, linear_peptide], name="itgavb3_linear_peptide_design")


In [ ]:
{
    "cyclic_helper_spec": cyclic_peptide.sequence,
    "linear_helper_spec": linear_peptide.sequence,
    "rgd_reference_1l5g": RGD_REFERENCE_1L5G,
}


## Run one design pass (cyclic peptide)

`fold()` produces a structural/design hypothesis for prioritization.

> Guardrail: treat these outputs as ranking signals; experimental validation is still required.


In [ ]:
result_cyclic = cyclic_design.fold()

result_cyclic


In [ ]:
result_cyclic.binder_specs


In [ ]:
result_cyclic.chain_design_summary()


## Optional follow-up

Compare with the linear branch:

```python
result_linear = linear_design.fold()
result_linear.binder_specs
result_linear.chain_design_summary()
```


## Science references

- RCSB PDB 1L5G (integrin alphaVbeta3 in complex with RGD ligand): https://www.rcsb.org/structure/1L5G
- Xiong et al., *Science* (2002), alphaVbeta3-RGD complex structure: https://pubmed.ncbi.nlm.nih.gov/11884718/
- Stupp et al., *Lancet Oncology* (2014), CENTRIC phase 3 cilengitide trial: https://pubmed.ncbi.nlm.nih.gov/25163906/
- FDA preclinical research basics (why experimental validation remains required): https://www.fda.gov/patients/drug-development-process/step-2-preclinical-research
